In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data.csv")
print("Фрагмент данных:")
print(df.head())
print()

trajectory = df["x"].to_numpy()
states = np.unique(trajectory)
print("Наблюдаемые состояния:", states)
print("Длина траектории:", len(trajectory))
print()

# -----------------------------
# 2. Подсчёт матрицы переходов n_ij
# -----------------------------
# Индексация: состояния от 1 до max_state (в данных встречаются 1..5)
max_state = int(states.max())
n = np.zeros((max_state, max_state), dtype=int)

for x_t, x_next in zip(trajectory[:-1], trajectory[1:]):
    n[x_t - 1, x_next - 1] += 1

print("Матрица частот переходов n_ij (строки — из состояния, столбцы — в состояние):")
print(pd.DataFrame(n, index=[f"i={i}" for i in range(1, max_state + 1)],
                     columns=[f"j={j}" for j in range(1, max_state + 1)]))
print()

# -----------------------------
# 3. ММП-оценка P
# -----------------------------
P_hat = np.zeros_like(n, dtype=float)

for i in range(max_state):
    row_sum = n[i].sum()
    if row_sum > 0:
        P_hat[i] = n[i] / row_sum
    else:
        # Переходов из этого состояния не наблюдалось
        P_hat[i] = np.nan  # или np.zeros(max_state)

print("ММП-оценка матрицы P (NaN = строка не наблюдалась):")
P_df = pd.DataFrame(
    P_hat,
    index=[f"i={i}" for i in range(1, max_state + 1)],
    columns=[f"j={j}" for j in range(1, max_state + 1)],
)
print(P_df.round(4))
print()

# -----------------------------
# 4. Вариант с доопределением ненаблюдаемых строк
#    (равномерное распределение по всем состояниям)
# -----------------------------
P_hat_filled = P_hat.copy()
for i in range(max_state):
    if np.isnan(P_hat_filled[i]).any():
        P_hat_filled[i] = np.ones(max_state) / max_state

print("ММП-оценка с доопределением ненаблюдаемых строк (равномерно):")
P_filled_df = pd.DataFrame(
    P_hat_filled,
    index=[f"i={i}" for i in range(1, max_state + 1)],
    columns=[f"j={j}" for j in range(1, max_state + 1)],
)
print(P_filled_df.round(4))

Фрагмент данных:
   t  x
0  0  5
1  1  3
2  2  5
3  3  1
4  4  5

Наблюдаемые состояния: [1 2 3 4 5]
Длина траектории: 43

Матрица частот переходов n_ij (строки — из состояния, столбцы — в состояние):
     j=1  j=2  j=3  j=4  j=5
i=1    0    0    0    0    3
i=2    1    1    2    0    1
i=3    0    4    1    1   10
i=4    0    1    0    0    0
i=5    2    0   13    0    2

ММП-оценка матрицы P (NaN = строка не наблюдалась):
        j=1   j=2     j=3     j=4     j=5
i=1  0.0000  0.00  0.0000  0.0000  1.0000
i=2  0.2000  0.20  0.4000  0.0000  0.2000
i=3  0.0000  0.25  0.0625  0.0625  0.6250
i=4  0.0000  1.00  0.0000  0.0000  0.0000
i=5  0.1176  0.00  0.7647  0.0000  0.1176

ММП-оценка с доопределением ненаблюдаемых строк (равномерно):
        j=1   j=2     j=3     j=4     j=5
i=1  0.0000  0.00  0.0000  0.0000  1.0000
i=2  0.2000  0.20  0.4000  0.0000  0.2000
i=3  0.0000  0.25  0.0625  0.0625  0.6250
i=4  0.0000  1.00  0.0000  0.0000  0.0000
i=5  0.1176  0.00  0.7647  0.0000  0.1176
